# Ranker Demo: BM25 + Semantic + Priors

Run the cells in order to test the `ranker` module using sample documents.

This notebook demonstrates:
- Lexical ranking with BM25
- Optional semantic influence via cosine similarity
- Popularity and recency priors
- k edge cases and simple metrics (NDCG@K, MRR)


In [2]:
import os, sys, platform
sys.path.append(os.path.abspath('../src'))
from ranker import rank, ndcg_at_k, mrr
from data.sample_data import get_sample_documents
print(f'Python: {platform.python_version()}')


Python: 3.12.8


In [13]:
docs = get_sample_documents()
len(docs), docs


(5,
 [{'id': 'doc_001',
   'text': 'Blue resume template modern professional',
   'tokens': ['blue', 'resume', 'template', 'modern', 'professional'],
   'clicks': 120,
   'age_days': 10},
  {'id': 'doc_002',
   'text': 'Wedding invitation floral theme',
   'tokens': ['wedding', 'invitation', 'floral', 'theme'],
   'clicks': 80,
   'age_days': 5},
  {'id': 'doc_003',
   'text': 'Business card minimalist design',
   'tokens': ['business', 'card', 'minimalist', 'design'],
   'clicks': 50,
   'age_days': 20},
  {'id': 'doc_004',
   'text': 'Resume template clean layout',
   'tokens': ['resume', 'template', 'clean', 'layout'],
   'clicks': 200,
   'age_days': 60},
  {'id': 'doc_005',
   'text': 'Birthday invitation fun colorful',
   'tokens': ['birthday', 'invitation', 'fun', 'colorful'],
   'clicks': 30,
   'age_days': 2}])

### Lexical ranking (BM25)


In [8]:
results_lex = rank('resume template', docs, k=5)
results_lex


[('doc_004', 1.5049076706454487),
 ('doc_001', 1.3233429194476165),
 ('doc_002', -0.7175853414141935),
 ('doc_003', -1.03998652019166),
 ('doc_005', -1.070678728487212)]

### Priors-only behavior (empty query)


In [9]:
results_priors = rank('', docs, k=3)
results_priors


[('doc_001', 0.2070480494988304),
 ('doc_004', 0.1771933177518169),
 ('doc_002', 0.09708439953327919)]

### Semantic influence (when embeddings are provided)


In [10]:
docs_sem = [
    {'id': 'a', 'tokens': ['irrelevant'], 'emb': [1.0, 0.0]},
    {'id': 'b', 'tokens': ['irrelevant'], 'emb': [0.0, 1.0]},
]
results_sem = rank('', docs_sem, k=2, weights=(0.0, 1.0, 0.0), query_embedding=[1.0, 0.0])
results_sem


[('a', 1.0), ('b', -1.0)]

### k edge cases


In [11]:
rank('resume', docs, k=0), len(rank('resume', docs, k=999))


([], 5)

### Simple metrics (NDCG@K, MRR)


In [12]:
gains = [3.0, 2.0, 1.0]
ndcg_at_k(gains, 3), mrr([0, 0, 1, 0])


(1.0, 0.3333333333333333)

In [3]:
from __future__ import annotations
import sys
from typing import Sequence

if "src" not in sys.path:
    sys.path.append("src")

from ranker import ndcg_at_k, mrr
print("Imports OK: ndcg_at_k, mrr available")

Imports OK: ndcg_at_k, mrr available


In [4]:

# Sample queries with graded gains (for NDCG) and binary relevances (for MRR)
queries = [
    {"id": "q1", "gains": [3, 2, 1, 0], "relevances": [1, 0, 0, 0]},
    {"id": "q2", "gains": [0, 1, 2, 3], "relevances": [0, 0, 1, 0]},
    {"id": "q3", "gains": [2, 0, 1], "relevances": [0, 1, 0]},
]

k = 3
per_query = []
for q in queries:
    nd = ndcg_at_k(q["gains"], k)
    mr = mrr(q["relevances"])
    per_query.append((q["id"], nd, mr))

avg_ndcg = sum(nd for _, nd, _ in per_query) / len(per_query) if per_query else 0.0
avg_mrr = sum(mr for _, _, mr in per_query) / len(per_query) if per_query else 0.0

print(f"Per-query metrics (id, ndcg@{k}, mrr):")
for pid, nd, mr in per_query:
    print(f"{pid}: {nd:.4f}, {mr:.4f}")
print("Averages:")
print(f"mean ndcg@{k}: {avg_ndcg:.4f}")
print(f"mean mrr: {avg_mrr:.4f}")


Per-query metrics (id, ndcg@3, mrr):
q1: 1.0000, 1.0000
q2: 0.3425, 0.3333
q3: 0.9502, 0.5000
Averages:
mean ndcg@3: 0.7642
mean mrr: 0.6111
